# 07 Team-Match-Basisdataset mit Spark bauen

Dieses Notebook implementiert BD-16. Es liest die in BD-06 erzeugten StatsBomb-Teammetriken, ergaenzt kanonische Match-Metadaten und verbindet jede Team-Zeile mit der passenden Gegner-Zeile.

Output:

- `data/silver/team_match_base.parquet/`

## Methodischer Ansatz

Der Schritt folgt der Spark-DataFrame-Logik aus den Kursunterlagen in `resources/Spark DataFrame.pdf`: relationale Transformationen wie `select`, `join` und aggregierte Qualitaetschecks werden deklarativ formuliert, statt Zeilen manuell zu iterieren. Fachlich bleibt die Analyseebene `team_match`: Ein reales Spiel erzeugt genau zwei Beobachtungen, eine pro Team.

Die Teammetriken kommen aus `data/bronze/team_match_statsbomb_metrics.parquet`. Match-Metadaten werden zusaetzlich aus `data/bronze/statsbomb_matches.parquet` gelesen, damit Turnier, Datum, Spielort und Laenderinformationen aus der kanonischen Match-Tabelle stammen. Die Gegnerinformationen werden ueber einen Self-Join auf `match_id` und `opponent_team_id` validiert und um zentrale Gegnermetriken erweitert.

In [ ]:
import os
import shutil

from project_paths import ensure_pyspark_path, project_root as get_project_root

ensure_pyspark_path()

from pyspark.sql import SparkSession, functions as F

SPARK_MASTER = os.getenv('SPARK_MASTER', 'local[*]')

base_path = get_project_root()
bronze_path = base_path / 'data' / 'bronze'
silver_path = base_path / 'data' / 'silver'
metrics_path = bronze_path / 'team_match_statsbomb_metrics.parquet'
matches_path = bronze_path / 'statsbomb_matches.parquet'
output_path = silver_path / 'team_match_base.parquet'

spark = (
    SparkSession.builder
    .appName('football-weather-bd16-team-match-base')
    .master(SPARK_MASTER)
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')

{
    'metrics_path': str(metrics_path),
    'matches_path': str(matches_path),
    'output_path': str(output_path),
    'spark_master': SPARK_MASTER,
}


## Bronze-Daten lesen

`team_match_statsbomb_metrics.parquet` enthaelt eine Zeile pro Team und Spiel. `statsbomb_matches.parquet` enthaelt die kanonischen Match- und Venue-Metadaten, die fuer spaetere Wetter- und Elo-Joins gebraucht werden.

In [ ]:
team_metrics = spark.read.parquet(str(metrics_path))
matches = spark.read.parquet(str(matches_path))

input_counts = {
    'team_metric_rows': team_metrics.count(),
    'team_metric_matches': team_metrics.select('match_id').distinct().count(),
    'match_rows': matches.count(),
    'match_ids': matches.select('match_id').distinct().count(),
}

team_metrics.select(
    'match_id', 'team_id', 'team_name', 'opponent_team_id', 'opponent_team_name', 'xg', 'shots', 'passes'
).orderBy('match_id', F.desc('is_home')).show(6, truncate=False)

input_counts

## Match-Metadaten normalisieren

Die Silver-Tabelle nutzt eindeutige, joinbare Feldnamen. Venue-Spalten bleiben erhalten, weil Wetterdaten spaeter ueber `match_id` und Spielort plausibilisiert werden.

In [ ]:
match_metadata = matches.select(
    F.col('match_id').cast('long').alias('match_id'),
    'scope_id',
    'short_name',
    F.col('competition_id').cast('long').alias('competition_id'),
    F.col('season_id').cast('long').alias('season_id'),
    'competition_name',
    'statsbomb_competition_name',
    'season_name',
    'match_date',
    'kick_off',
    F.col('match_week').cast('long').alias('match_week'),
    'competition_stage_name',
    F.col('stadium_id').cast('long').alias('stadium_id'),
    'stadium_name',
    F.col('stadium_country_id').cast('long').alias('stadium_country_id'),
    'stadium_country_name',
    F.col('home_team_home_team_id').cast('long').alias('home_team_id'),
    F.col('home_team_home_team_name').alias('home_team_name'),
    F.col('away_team_away_team_id').cast('long').alias('away_team_id'),
    F.col('away_team_away_team_name').alias('away_team_name'),
    F.col('home_score').cast('long').alias('home_score'),
    F.col('away_score').cast('long').alias('away_score'),
)

match_metadata.orderBy('match_id').show(5, truncate=False)

## Team-Zeilen und Gegner-Zeilen verbinden

Der Self-Join sucht innerhalb desselben `match_id` die Zeile, deren `team_id` zur `opponent_team_id` der aktuellen Zeile passt. Dadurch wird nicht nur ein Gegnername uebernommen, sondern die Gegenzeile explizit nachgewiesen.

In [ ]:
metric_columns = [
    'xg',
    'shots',
    'xg_per_shot',
    'passes',
    'successful_passes',
    'pass_completion_rate',
    'pressures',
    'counterpressures',
    'duels',
    'carries',
    'long_balls',
    'long_ball_share',
]

team_rows = team_metrics.select(
    F.col('match_id').cast('long').alias('match_id'),
    F.col('team_id').cast('long').alias('team_id'),
    'team_name',
    F.col('opponent_team_id').cast('long').alias('opponent_team_id'),
    'opponent_team_name',
    F.col('is_home').cast('boolean').alias('is_home'),
    F.col('team_score').cast('long').alias('team_score'),
    F.col('opponent_score').cast('long').alias('opponent_score'),
    *[F.col(column) for column in metric_columns],
)

opponent_rows = team_rows.select(
    F.col('match_id').alias('opponent_match_id'),
    F.col('team_id').alias('joined_opponent_team_id'),
    F.col('team_name').alias('joined_opponent_team_name'),
    F.col('opponent_team_id').alias('joined_opponent_opponent_team_id'),
    F.col('team_score').alias('joined_opponent_score'),
    F.col('opponent_score').alias('joined_opponent_score_against'),
    *[F.col(column).alias(f'opponent_{column}') for column in metric_columns],
)

team_with_opponent = (
    team_rows.alias('team')
    .join(
        opponent_rows.alias('opponent'),
        (F.col('team.match_id') == F.col('opponent.opponent_match_id'))
        & (F.col('team.opponent_team_id') == F.col('opponent.joined_opponent_team_id')),
        'left',
    )
    .drop('opponent_match_id')
)

team_with_opponent.select(
    'match_id', 'team_name', 'opponent_team_name', 'joined_opponent_team_name', 'xg', 'opponent_xg'
).orderBy('match_id', F.desc('is_home')).show(6, truncate=False)

## Silver-Basisdataset bauen und speichern

Das Ergebnis verbindet Match-Metadaten, Teammetriken und Gegnermetriken in einer stabilen Spaltenreihenfolge. `goal_diff` und `match_result` sind einfache Team-Perspektiven-Features fuer spaetere Analysen.

In [ ]:
team_match_base = (
    team_with_opponent
    .join(match_metadata, on='match_id', how='left')
    .withColumn('goal_diff', F.col('team_score') - F.col('opponent_score'))
    .withColumn(
        'match_result',
        F.when(F.col('goal_diff') > 0, F.lit('win'))
        .when(F.col('goal_diff') < 0, F.lit('loss'))
        .otherwise(F.lit('draw')),
    )
    .select(
        'match_id',
        'scope_id',
        'short_name',
        'competition_id',
        'season_id',
        'competition_name',
        'statsbomb_competition_name',
        'season_name',
        'match_date',
        'kick_off',
        'match_week',
        'competition_stage_name',
        'stadium_id',
        'stadium_name',
        'stadium_country_id',
        'stadium_country_name',
        'home_team_id',
        'home_team_name',
        'away_team_id',
        'away_team_name',
        'home_score',
        'away_score',
        'team_id',
        'team_name',
        'opponent_team_id',
        'opponent_team_name',
        'is_home',
        'team_score',
        'opponent_score',
        'goal_diff',
        'match_result',
        *metric_columns,
        *[f'opponent_{column}' for column in metric_columns],
    )
    .orderBy('match_id', F.desc('is_home'))
)

if output_path.exists():
    if output_path.is_dir():
        shutil.rmtree(output_path)
    else:
        output_path.unlink()

team_match_base.write.mode('overwrite').parquet(str(output_path))

written = spark.read.parquet(str(output_path))
{
    'written_rows': written.count(),
    'written_matches': written.select('match_id').distinct().count(),
    'output_path': 'data/silver/team_match_base.parquet',
}

## Qualitaetschecks

Die Akzeptanzkriterien werden als Assertions geprueft: jede Zeile ist ein Team in einem Spiel, jedes Spiel hat genau zwei Team-Zeilen, Gegnerinformationen sind vollstaendig und der Output wurde mit Spark als Parquet geschrieben.

In [ ]:
required_non_null_columns = [
    'match_id',
    'team_id',
    'team_name',
    'opponent_team_id',
    'opponent_team_name',
    'joined_opponent_team_id',
    'joined_opponent_team_name',
    'match_date',
    'competition_name',
    'stadium_name',
]

row_count = team_match_base.count()
match_count = matches.select('match_id').distinct().count()
duplicate_team_match_rows = (
    team_match_base
    .groupBy('match_id', 'team_id')
    .count()
    .filter(F.col('count') != 1)
    .count()
)
non_two_team_matches = (
    team_match_base
    .groupBy('match_id')
    .count()
    .filter(F.col('count') != 2)
    .count()
)
missing_required_values = team_with_opponent.join(match_metadata, on='match_id', how='left').select(
    [F.sum(F.col(column).isNull().cast('int')).alias(column) for column in required_non_null_columns]
).collect()[0].asDict()
opponent_name_mismatch_count = team_with_opponent.filter(
    F.col('joined_opponent_team_name') != F.col('opponent_team_name')
).count()
opponent_reverse_link_mismatch_count = team_with_opponent.filter(
    F.col('joined_opponent_opponent_team_id') != F.col('team_id')
).count()
score_symmetry_mismatch_count = team_with_opponent.filter(
    (F.col('joined_opponent_score') != F.col('opponent_score'))
    | (F.col('joined_opponent_score_against') != F.col('team_score'))
).count()

quality_checks = {
    'matches': match_count,
    'team_match_rows': row_count,
    'expected_team_match_rows': match_count * 2,
    'duplicate_team_match_rows': duplicate_team_match_rows,
    'non_two_team_matches': non_two_team_matches,
    'missing_required_values': missing_required_values,
    'opponent_name_mismatch_count': opponent_name_mismatch_count,
    'opponent_reverse_link_mismatch_count': opponent_reverse_link_mismatch_count,
    'score_symmetry_mismatch_count': score_symmetry_mismatch_count,
}

assert quality_checks['team_match_rows'] == quality_checks['expected_team_match_rows']
assert quality_checks['duplicate_team_match_rows'] == 0
assert quality_checks['non_two_team_matches'] == 0
assert all(value == 0 for value in missing_required_values.values())
assert quality_checks['opponent_name_mismatch_count'] == 0
assert quality_checks['opponent_reverse_link_mismatch_count'] == 0
assert quality_checks['score_symmetry_mismatch_count'] == 0

quality_checks

In [ ]:
written = spark.read.parquet(str(output_path))
assert written.count() == row_count
assert written.select('match_id', 'team_id').distinct().count() == row_count

written.select(
    'match_id',
    'short_name',
    'match_date',
    'stadium_name',
    'team_name',
    'opponent_team_name',
    'team_score',
    'opponent_score',
    'xg',
    'opponent_xg',
).orderBy('match_id', F.desc('is_home')).show(10, truncate=False)

BD-16 ist erfuellt, wenn die Assertions erfolgreich sind und `data/silver/team_match_base.parquet/` existiert. Das Dataset enthaelt pro Spiel genau zwei Team-Match-Zeilen, kanonische Match-Metadaten und validierte Gegnerinformationen inklusive ausgewaehlter Gegnermetriken.